# Minimal Example for Calculating the Elastic Response of a Beam

In [ ]:
from sectionproperties.analysis import Section
from sectionproperties.pre.library import i_section
from sectionproperties.pre import Material
from pydantic import BaseModel, Field

In [6]:
# define material properties for structural steel
material = Material(
    name="Structural Steel",
    elastic_modulus=210e3,   # in MPa
    poissons_ratio=0.3,      # dimensionless
    yield_strength=250,      # in MPa
    density=7.85e-6,          # in kg/mm^3
    color="gray"
)

# define I-section geometry
geometry = i_section(
    d=300,                  # depth in mm
    b=150,                  # flange width in mm
    t_f=15,                 # flange thickness in mm
    t_w=10,                 # web thickness in mm
    r=12,                   # root radius in mm
    n_r=8,                  # number of points to define the root radius
)

# assign material to geometry
geometry.material=material

# generate finite element mesh
geometry.create_mesh(mesh_sizes=[4])  # mesh size in mm

# create section object for analysis
section = Section(geometry=geometry)

In [ ]:
class SectionStiffnessProperties(BaseModel):
    """
    Data class representing key stiffness properties of a cross section.
    """
    A: float = Field(..., description="Area of the section with SI units [m^2].", gt=0)
    ksx: float = Field(..., description="Shear correction factor in x direction (dimensionless).", gt=0)
    ksy: float = Field(..., description="Shear correction factor in y direction (dimensionless).", gt=0)
    Ixx: float = Field(..., description="Second moment of area about x-axis with SI units [m^4].", gt=0)
    Iyy: float = Field(..., description="Second moment of area about y-axis with SI units [m^4].", gt=0)
    Ixy: float = Field(..., description="Product moment of area with SI units [m^4].", gt=0)


def calculate_section_properties(section: Section) -> dict:
    """Extract key geometric properties of the section."""

    # calculate geometric and warping properties
    section.calculate_geometric_properties()
    section.calculate_warping_properties()

    # retrieve properties
    area = section.get_area()
    shear_area_x, shear_area_y = section.get_eas(e_ref=material)
    Ixx, Iyy, Ixy = section.get_eic(e_ref=material)

    ksx = shear_area_x / area
    ksy = shear_area_y / area

    return SectionStiffnessProperties(
        A=area,
        ksx=ksx,
        ksy=ksy,
        Ixx=Ixx,
        Iyy=Iyy,
        Ixy=Ixy,
    )
    
    
props: SectionStiffnessProperties = calculate_section_properties(section)

SectionStiffnessProperties(A=7327.397797144078, ksx=0.5447893243348102, ksy=0.39155218714420476, Ixx=110094531.791528, Iyy=8468301.22926223, Ixy=4.929315476190476e-07)